# Kaggle Runner: Pointwise PURS (MovieLens-1M)

This notebook runs your updated binary pointwise PURS pipeline end-to-end on Kaggle.

What it does:
- Sets up repository path in a writable location
- Installs dependencies
- Downloads/checks MovieLens-1M into data/raw/ml-1m
- Runs ExperimentRunner with pointwise settings
- Prints strict public-style PURS metrics: AUC, Hit Rate, Precision, Coverage, Unexpectedness

In [ ]:
from pathlib import Path
import os
import subprocess

working_repo = Path('/kaggle/working/recsys')

if not working_repo.exists():
    print('Cloning recsys into /kaggle/working/recsys ...')
    subprocess.check_call(
        ['git', 'clone', 'https://github.com/codetuscan/recsys.git', str(working_repo)]
    )
else:
    print('Repo exists, pulling latest changes...')
    subprocess.check_call(
        ['git', '-C', str(working_repo), 'pull']
    )

os.chdir(working_repo)
print('Working directory:', Path.cwd())

In [ ]:
from pathlib import Path
import os
import sys
import shutil

candidates = [
    Path('/kaggle/working/recsys'),
    Path('/kaggle/input/recsys'),
    Path('/kaggle/input/recsys/recsys'),
    Path.cwd(),
]

repo_root = None
for candidate in candidates:
    if (candidate / 'config' / 'base_config.py').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the recsys repository. Put it in /kaggle/working/recsys or /kaggle/input/recsys.'
    )

if str(repo_root).startswith('/kaggle/input'):
    writable_root = Path('/kaggle/working/recsys')
    if not writable_root.exists():
        shutil.copytree(repo_root, writable_root)
    repo_root = writable_root

os.chdir(repo_root)
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

print('Repository root:', repo_root)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys
from pathlib import Path

requirements_file = Path('requirements-kaggle.txt')
if not requirements_file.exists():
    requirements_file = Path('requirements.txt')

print('Installing dependencies from', requirements_file)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)])
print('Dependency install done.')

In [ ]:
from pathlib import Path
import shutil

raw_dir = Path('data/raw')
ml1m_dir = raw_dir / 'ml-1m'
required_files = ['ratings.dat', 'movies.dat', 'users.dat']
optional_files = ['README']

def has_required_files(folder: Path) -> bool:
    return all((folder / name).exists() for name in required_files)

if has_required_files(ml1m_dir):
    print('MovieLens-1M already available at', ml1m_dir)
else:
    input_root = Path('/kaggle/input')
    candidate_dirs = []

    if input_root.exists():
        for ratings_path in input_root.glob('**/ratings.dat'):
            folder = ratings_path.parent
            if has_required_files(folder):
                candidate_dirs.append(folder)

    if not candidate_dirs:
        raise FileNotFoundError(
            'Could not find MovieLens-1M in Kaggle Input. Add a dataset with ratings.dat, movies.dat, and users.dat.'
        )

    src_dir = sorted(candidate_dirs, key=lambda p: len(str(p)))[0]
    print('Using Kaggle Input dataset from', src_dir)

    raw_dir.mkdir(parents=True, exist_ok=True)
    ml1m_dir.mkdir(parents=True, exist_ok=True)

    for name in required_files + optional_files:
        src = src_dir / name
        dst = ml1m_dir / name
        if src.exists():
            shutil.copy2(src, dst)

for name in required_files:
    path = ml1m_dir / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required file after copy: {path}')

print('MovieLens-1M is ready at', ml1m_dir)

## Run Mode
- Set `SMOKE_RUN = True` for fast validation
- Set `SMOKE_RUN = False` for full baseline (100 epochs)

In [ ]:
from datetime import datetime

SMOKE_RUN = True

subset = 0.02 if SMOKE_RUN else None
epochs = 1 if SMOKE_RUN else 100
run_tag = 'smoke' if SMOKE_RUN else 'full'
exp_name = f'kaggle_purs_pointwise_{run_tag}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print('SMOKE_RUN:', SMOKE_RUN)
print('subset:', subset)
print('epochs:', epochs)
print('experiment:', exp_name)

In [ ]:
from pathlib import Path
from recsys.config import load_config
from recsys.experiments.experiment_runner import ExperimentRunner

config = load_config('kaggle')

# Model/task selection
config.model.model_name = 'purs'
config.data.dataset_name = 'movielens-1m'
config.data.positive_rating_threshold = 3.5

# Baseline hyperparameters
config.model.epochs = epochs
config.model.batch_size = 256
config.model.learning_rate = 0.01
config.model.embedding_dim = 128
config.model.gru_hidden_dim = 128
config.model.dropout = 0.1

# Data/eval settings
config.data.data_subset = subset
config.experiment.k_values = [10]
config.experiment.eval_every = 1

# Keep full runs from stopping early before all planned epochs.
config.experiment.early_stopping_patience = epochs + 1

# Force raw_data path to prepared local copy from Cell 5 (avoids Kaggle slug/path mismatch).
local_ml1m = Path('data/raw/ml-1m').resolve()
if not local_ml1m.exists():
    raise FileNotFoundError(
        f'Expected prepared dataset at {local_ml1m}. Run Cell 5 first to copy MovieLens files from Kaggle Input.'
    )
config.paths.raw_data = local_ml1m
print('Using raw data path:', config.paths.raw_data)

# Runtime controls
config.experiment.experiment_name = exp_name
config.model.num_workers = 2

runner = ExperimentRunner(config)
results = runner.run()
results

In [ ]:
public_metric_map = {
    'auc': 'AUC',
    'hit_rate': 'Hit Rate (score > 0.5)',
    'precision': 'Precision (same public hit-rate formula)',
    'coverage': 'Coverage',
    'unexpectedness': 'Unexpectedness',
}

print('Public-style PURS metric summary')
print(f"best_selection_metric: {results.get('best_selection_metric', 'auc')}")
if 'best_selection_score' in results:
    print(f"best_selection_score: {results['best_selection_score']:.6f}")

for key, label in public_metric_map.items():
    if key in results:
        print(f"{label} ({key}): {results[key]:.6f}")

print('\nLog directory:', runner.metrics_logger.get_experiment_dir())

## Legacy Public-Style PURS Runner (Kaggle)

This section runs the isolated TensorFlow-compatible public-style pipeline using the same repository cloned above.

Run Cells 2 and 3 first so the notebook is inside the repository root.

In [ ]:
import subprocess
import sys
from pathlib import Path

legacy_requirements = Path('legacy_public_purs/requirements-kaggle-legacy.txt')
if not legacy_requirements.exists():
    raise FileNotFoundError(f'Legacy requirements file not found: {legacy_requirements}')

print('Installing legacy dependencies from', legacy_requirements)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(legacy_requirements)])
print('Legacy dependency install done.')

## Legacy Run Mode
- Set LEGACY_SMOKE_RUN = True for a quick check (1 epoch, subset 0.02).
- Set LEGACY_SMOKE_RUN = False to use your config values exactly.

In [ ]:
from datetime import datetime
from pathlib import Path
import yaml

LEGACY_SMOKE_RUN = False

legacy_output_dir = Path('/kaggle/working/legacy_public_purs')
legacy_output_dir.mkdir(parents=True, exist_ok=True)

base_cfg_path = Path('config/kaggle_config.yaml')
if not base_cfg_path.exists():
    raise FileNotFoundError(f'Config not found: {base_cfg_path}')

with open(base_cfg_path, 'r') as f:
    legacy_cfg = yaml.safe_load(f)

if LEGACY_SMOKE_RUN:
    legacy_cfg.setdefault('data', {})['data_subset'] = 0.02
    legacy_cfg.setdefault('model', {})['epochs'] = 1

legacy_cfg.setdefault('experiment', {})['experiment_name'] = (
    f"legacy_public_purs_{'smoke' if LEGACY_SMOKE_RUN else 'full'}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
 )

legacy_cfg_path = legacy_output_dir / 'legacy_kaggle_config.yaml'
with open(legacy_cfg_path, 'w') as f:
    yaml.safe_dump(legacy_cfg, f, sort_keys=False)

print('LEGACY_SMOKE_RUN:', LEGACY_SMOKE_RUN)
print('Legacy config path:', legacy_cfg_path)
print('batch_size:', legacy_cfg['model']['batch_size'])
print('epochs:', legacy_cfg['model']['epochs'])
print('learning_rate:', legacy_cfg['model']['learning_rate'])
print('history_length:', legacy_cfg['model']['history_length'])
print('dataset_name:', legacy_cfg['data']['dataset_name'])

In [ ]:
import shlex
import subprocess
import sys

legacy_script = Path('legacy_public_purs/train_public_style.py')
if not legacy_script.exists():
    raise FileNotFoundError(f'Legacy training script not found: {legacy_script}')

cmd = [
    sys.executable,
    str(legacy_script),
    '--config', str(legacy_cfg_path),
    '--output-dir', str(legacy_output_dir),
]

print('Running command:')
print(' '.join(shlex.quote(str(x)) for x in cmd))
subprocess.check_call(cmd)

In [ ]:
import json
from pathlib import Path

metrics_path = Path(legacy_output_dir) / 'metrics.json'
if not metrics_path.exists():
    raise FileNotFoundError(f'Metrics file not found: {metrics_path}')

with open(metrics_path, 'r') as f:
    legacy_metrics = json.load(f)

print('Legacy public-style run summary')
print('dataset_name:', legacy_metrics.get('dataset_name'))
print('batch_size:', legacy_metrics.get('batch_size'))
print('epochs:', legacy_metrics.get('epochs'))
print('history_length:', legacy_metrics.get('history_length'))

epoch_metrics = legacy_metrics.get('epoch_metrics', [])
if epoch_metrics:
    last = epoch_metrics[-1]
    print('\nLast epoch metrics:')
    print('train_auc:', last.get('train_auc'))
    print('eval_auc:', last.get('eval_auc'))
    print('hit_rate:', last.get('hit_rate'))
    print('coverage:', last.get('coverage'))
    print('unexpectedness:', last.get('unexpectedness'))
    print('learning_rate:', last.get('lr'))
else:
    print('No epoch metrics found in metrics.json')

print('\nMetrics path:', metrics_path)